# VLM profiling

In [1]:
import time
import mlx.core as mx
from mlx_vlm import load
from mlx_vlm.prompt_utils import apply_chat_template
from mlx_vlm.utils import load_config, prepare_inputs
from mlx_vlm.models.cache import make_prompt_cache
from PIL import Image

/Users/harini/Documents/GitHub/zing-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model, processor = load("mlx-community/Qwen2-VL-2B-Instruct-4bit")
config = load_config("mlx-community/Qwen2-VL-2B-Instruct-4bit")

Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 58180.76it/s]
The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 118300.88it/s]


## timing breakdown
averaged after 10 trials

In [19]:
N = 10
prompt = apply_chat_template(processor, config, "what is this?", num_images=1)

times = []
for i in range(N):
    t0 = time.perf_counter()
    inputs = prepare_inputs(processor, images=["example.jpg"], prompts=prompt)
    times.append(time.perf_counter() - t0)

print(f"prepare_inputs: {sum(times)/N*1000:.0f}ms (avg of {N})")

input_ids = inputs["input_ids"]
pixel_values = inputs["pixel_values"]
mask = inputs["attention_mask"]
grid_thw = inputs["image_grid_thw"]

prepare_inputs: 22ms (avg of 10)


In [20]:
dtype = model.vision_tower.patch_embed.proj.weight.dtype

times = []
for i in range(N):
    t0 = time.perf_counter()
    hidden = model.vision_tower(pixel_values.astype(dtype), grid_thw)
    mx.eval(hidden)
    times.append(time.perf_counter() - t0)

print(f"vision encoder: {sum(times)/N*1000:.0f}ms (avg of {N})")
print(f"  output shape: {hidden.shape}")

vision encoder: 2852ms (avg of 10)
  output shape: (1040, 1536)


In [21]:
times = []
for i in range(N):
    t0 = time.perf_counter()
    embeds = model.language_model.model.embed_tokens(input_ids)
    merged = model.merge_input_ids_with_image_features(
        model.config.image_token_id, model.config.video_token_id,
        hidden, embeds, input_ids
    )
    mx.eval(merged)
    times.append(time.perf_counter() - t0)

print(f"embed + merge: {sum(times)/N*1000:.0f}ms (avg of {N})")
print(f"  merged shape: {merged.shape}")

embed + merge: 123ms (avg of 10)
  merged shape: (1, 1065, 1536)


In [22]:
times = []
for i in range(N):
    t0 = time.perf_counter()
    cache = make_prompt_cache(model.language_model)
    out = model.language_model(input_ids, merged, mask=mask, cache=cache)
    mx.eval(out.logits)
    times.append(time.perf_counter() - t0)

print(f"LM prefill: {sum(times)/N*1000:.0f}ms (avg of {N})")
print(f"  logits shape: {out.logits.shape}")

LM prefill: 1728ms (avg of 10)
  logits shape: (1, 1065, 151936)


## kv cache

In [9]:
print(f"num layers: {len(cache)}")
print(f"tokens cached: {cache[0].offset}")
print(f"layer 0 keys shape: {cache[0].keys.shape}")
print(f"layer 0 keys dtype: {cache[0].keys.dtype}")

total_bytes = sum(c.keys.nbytes + c.values.nbytes for c in cache)
print(f"\ntotal KV cache: {total_bytes/1e6:.1f} MB")
print(f"bytes per token: {total_bytes/cache[0].offset:.0f}")

num layers: 28
tokens cached: 1065
layer 0 keys shape: (1, 2, 1280, 128)
layer 0 keys dtype: mlx.core.float16

total KV cache: 36.7 MB
bytes per token: 34460


## memory

In [10]:
print(f"active: {mx.get_active_memory()/1e9:.2f} GB")
print(f"peak: {mx.get_peak_memory()/1e9:.2f} GB")

active: 1.64 GB
peak: 4.65 GB


## image size experiment

In [23]:
import tempfile, os

img = Image.open("example.jpg")
print(f"original: {img.size}")

original: (740, 1110)


In [24]:
for size in [(256, 256), (512, 512), (768, 768)]:
    resized = img.resize(size)
    with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
        resized.save(f.name)
        
        inputs = prepare_inputs(processor, images=[f.name], prompts=prompt)
        pv = inputs["pixel_values"]
        gt = inputs["image_grid_thw"]
        
        times = []
        for i in range(N):
            t0 = time.perf_counter()
            h = model.vision_tower(pv.astype(dtype), gt)
            mx.eval(h)
            times.append(time.perf_counter() - t0)
        
        print(f"{size}: {h.shape[0]} tokens, {sum(times)/N*1000:.0f}ms avg")
        os.unlink(f.name)

(256, 256): 81 tokens, 667ms avg
(512, 512): 324 tokens, 824ms avg
(768, 768): 729 tokens, 2248ms avg


## quality vs size test

In [25]:
from mlx_vlm import generate

test_prompt = "Describe what you see in this image in detail."
formatted = apply_chat_template(processor, config, test_prompt, num_images=1)

for size in [(256, 256), (512, 512), (768, 768), None]:
    if size:
        resized = img.resize(size)
        with tempfile.NamedTemporaryFile(suffix='.jpg', delete=False) as f:
            resized.save(f.name)
            img_path = f.name
    else:
        img_path = "example.jpg"
        size = img.size
    
    t0 = time.perf_counter()
    out = generate(model, processor, formatted, [img_path], max_tokens=100, verbose=False)
    t = time.perf_counter() - t0
    
    print(f"\n{'='*50}")
    print(f"SIZE: {size}, TIME: {t*1000:.0f}ms")
    print(f"{'='*50}")
    print(out.text[:300] if hasattr(out, 'text') else out[:300])
    
    if size != img.size:
        os.unlink(img_path)


SIZE: (256, 256), TIME: 38986ms
The image depicts a silhouette of two hands forming a heart shape against a beautiful sunset sky. The hands are positioned in such a way that they overlap each other, creating a symmetrical heart shape. The background features a vivid and colorful sunset, with hues of orange, red, and yellow dominat

SIZE: (512, 512), TIME: 1973ms
The image depicts a heart shape formed by two silhouetted hands against a vibrant, colorful sky during a sunset. The hands are positioned in such a way that they form a heart shape, with the fingers of one hand touching the thumb of the other. The background features a gradient of colors, transition

SIZE: (768, 768), TIME: 3293ms
The image depicts a heart shape formed by two hands against a beautiful sunset sky. The hands are silhouetted against the vibrant colors of the sky, which includes shades of orange, red, and yellow. The sky appears to be either dawn or dusk, with the sun low on the horizon, casting a warm glow over 
